# Campos obrigatórios × opcionais

Até agora, todos os campos que definimos nos modelos eram obrigatórios.

Fazendo uma analogia: como tornamos opcional um argumento de função em Python?

Damos a ele um valor padrão (*default*).

O Pydantic usa a mesma abordagem.

Tornamos um campo opcional simplesmente dando um valor padrão na definição do campo.

Há algumas formas de fazer isso; veremos uma aqui e outras mais adiante no curso.

In [1]:
from pydantic import BaseModel

class Circle(BaseModel):
    center: tuple[int, int] = (0, 0)
    radius: int

Neste modelo, `center` é um campo **opcional**, que assume `(0, 0)` quando não vem nos dados desserializados. Já `radius`,
sem default, é **obrigatório**.

Também vemos isso inspecionando os campos do modelo:

In [2]:
print(f"Circle.model_fields: {Circle.model_fields!r}")

Circle.model_fields: {'center': FieldInfo(annotation=tuple[int, int], required=False, default=(0, 0)), 'radius': FieldInfo(annotation=int, required=True)}


Repare que `center` tem `required=False`, enquanto `radius` tem `required=True`.

Agora podemos criar instâncias de `Circle` sem informar `center`:

In [3]:
print(f"Circle(radius=1): {Circle(radius=1)!r}")

Circle(radius=1): Circle(center=(0, 0), radius=1)


Nenhuma exceção de validação, e o default correto está lá.

Funciona igual em todos os outros métodos de desserialização:

In [4]:
data = {"radius": 1}
data_json = '{"radius": 1}'

In [5]:
print(f"Circle.model_validate(data): {Circle.model_validate(data)!r}")

Circle.model_validate(data): Circle(center=(0, 0), radius=1)


In [6]:
print(f"Circle.model_validate_json(data_json): {Circle.model_validate_json(data_json)!r}")

Circle.model_validate_json(data_json): Circle(center=(0, 0), radius=1)


Claro que também podemos informar `center`:

In [7]:
print(f"Circle(center=(1, 1), radius=2): {Circle(center=(1, 1), radius=2)!r}")

Circle(center=(1, 1), radius=2): Circle(center=(1, 1), radius=2)


É preciso cuidado com uma coisa ao definir um valor padrão.

Quando fornecemos um valor, o Pydantic o valida antes de colocá-lo na instância. Porém, o comportamento padrão do Pydantic **não valida
valores default**!

Faz algum sentido: nós escrevemos a definição do modelo, então deveríamos fornecer apenas defaults válidos.

Então, o problema é este:

In [8]:
class Model(BaseModel):
    field: int = "Python"

O default é totalmente incompatível com `int` e, mesmo assim, funciona:

In [9]:
print(f"Model(): {Model()!r}")

Model(): Model(field='Python')


Portanto, cuidado com os defaults: a responsabilidade de fornecer defaults consistentes é nossa.

O Pydantic oferece uma forma de forçar a validação dos defaults — ao custo de um pouco de processamento a cada uso do default.
Algo que podemos evitar escrevendo código correto. Veremos depois como habilitar a validação de defaults (`validate_default`).

Normalmente em Python, inclusive em dataclasses, precisamos de cuidado extra quando o default é um objeto **mutável**.

Vejamos um exemplo com uma função Python comum.

Aliás, escrever uma função que modifica a entrada e a devolve é uma péssima prática — uso aqui só para ilustrar.

In [10]:
from time import time

def extend_list(user_list: list = []):
    user_list.append(int(time()))
    return user_list

Podemos chamar a função com a nossa própria lista:

In [11]:
my_times = []
extend_list(my_times)
print(f"my_times: {my_times!r}")

my_times: [1794395689]


Mas, como `user_list` tem default, também podemos chamar assim:

In [12]:
my_times = extend_list()
print(f"my_times: {my_times!r}")

my_times: [1794395689]


Parece ter funcionado.

Mas e isto?

In [13]:
my_new_times = extend_list()

Poderíamos esperar que `my_new_times` tivesse um único elemento: o horário (epoch) da chamada de `extend_list()`.

In [14]:
print(f"my_new_times: {my_new_times!r}")

my_new_times: [1794395689, 1794395689]


Hein??

Esse é o problema que mencionei. Os valores default de argumentos são calculados e guardados junto da função — não são recriados a
cada chamada. Ou seja, o default é **compartilhado** entre todas as chamadas.

Algo parecido acontece com dataclasses. Lá, contornamos o problema com uma *default factory*: em vez de um objeto mutável como default,
passamos uma função que é chamada a cada nova instância e, portanto, devolve um objeto **novo**.

O Pydantic também oferece *default factory* (veremos depois).

Mas ele vai além e permite simplesmente definir objetos mutáveis como default: o Pydantic **copia** o default a cada nova instância.

O jeito certo de escrever a função.

> 📝 **Nota de revisão:** o original propunha `user_list = user_list or []`. Isso tem um bug: se o chamador passar uma lista **vazia**,
> `[] or []` cria uma lista nova e a lista do chamador não é alterada (comportamento diferente de quando ela já tem itens).
> O teste correto é por identidade com `None`:

In [15]:
def extend_list_buggy(user_list: list | None = None):
    user_list = user_list or []
    user_list.append(int(time()))
    return user_list

my_times = []
extend_list_buggy(my_times)
print(f"my_times: {my_times!r}")                      # deveria ter 1 elemento, mas a lista do chamador continua vazia

my_times: []


In [16]:
def extend_list(user_list: list | None = None):
    if user_list is None:
        user_list = []
    user_list.append(int(time()))
    return user_list

In [17]:
my_times = []
extend_list(my_times)
print(f"len(my_times): {len(my_times)!r}")

len(my_times): 1


In [18]:
my_times = extend_list()
print(f"len(my_times): {len(my_times)!r}")

len(my_times): 1


In [19]:
my_new_times = extend_list()
print(f"len(my_new_times): {len(my_new_times)!r}")

len(my_new_times): 1
